# 2A. Implementación de un motor de búsqueda léxico y BM25

**Autor:** David Segundo García · **Curso:** Recuperación de información 2026-2

Este notebook construye desde cero un buscador léxico completo: preprocesamiento del texto,
índice invertido con frecuencias, ranking BM25 e interfaz de consulta con `ipywidgets`.

## Mapa del notebook

| Sección | Contenido |
|---|---|
| 0 | Dependencias y utilidades |
| 1 | Corpus: 120 documentos, 12 temáticas |
| 2 | `preprocesar()`: tokenización, minúsculas, puntuación, *stopwords*, *stemming* |
| 3 | `IndiceInvertido`: *postings* con `tf`, `df`, longitudes y `avgdl` |
| 4 | `BM25`: fórmula completa con longitud promedio de documentos |
| 5 | Interfaz `ipywidgets` con Top-5 y puntuación |
| 6 | Evaluación cuantitativa: casos de acierto |
| 7 | Caso de fallo por búsqueda exacta (sinónimos) y mitigación |
| 8 | Conclusiones (insumo del reporte en PDF) |

> **Reproducibilidad.** El corpus vive en `corpus/corpus.json` y se genera de forma determinista con
> `corpus/generar_corpus.py`. No se requiere conexión a internet: si `nltk` no está disponible, el
> notebook usa una lista de *stopwords* y un *stemmer* de respaldo implementados aquí mismo.

## 0. Dependencias y utilidades

In [ ]:
"""Motor de búsqueda léxico con BM25 — actividad 2A."""

from __future__ import annotations

import json
import math
import re
import sys
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Callable, Iterable, Sequence

import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)


Python 3.14.5
pandas 3.0.5


In [ ]:
# Dependencias opcionales. El notebook funciona con o sin ellas, pero `ipywidgets`
# es necesario para la interfaz gráfica del paso 5 de la actividad.
try:
    import ipywidgets as widgets
    from IPython.display import HTML, display

    HAY_WIDGETS = True
    print(f"ipywidgets {widgets.__version__} disponible")
except ImportError:  # pragma: no cover - depende del entorno
    HAY_WIDGETS = False
    print("ipywidgets NO disponible -> ejecuta: pip install ipywidgets")

try:
    import nltk

    HAY_NLTK = True
    print(f"nltk {nltk.__version__} disponible")
except ImportError:  # pragma: no cover - depende del entorno
    HAY_NLTK = False
    print("nltk NO disponible -> se usarán las implementaciones de respaldo")

ipywidgets 8.1.9 disponible


nltk 3.10.3 disponible


## 1. El corpus

**Paso 1 de la actividad.** Colección curada de **120 documentos** en español repartidos en
**12 temáticas** (inteligencia artificial, astronomía, salud, gastronomía, deportes, economía,
medio ambiente, historia, arte y música, transporte, ciberseguridad, viajes).

La diversidad temática es intencional: permite comprobar que el ranking discrimina entre dominios
y no se limita a contar palabras. También lo es el vocabulario elegido — el corpus usa
*automóvil*, *médico* y *computadora*, y **nunca** sus sinónimos *coche*, *doctor* u *ordenador*.
Esa asimetría se explota en la sección 7 para exhibir el límite de la búsqueda exacta.

In [ ]:
def cargar_corpus(ruta: Path | None = None) -> list[dict]:
    """Carga la colección de documentos desde disco.

    Args:
        ruta: Ubicación del JSON del corpus. Si es ``None`` se busca en
            ``corpus/corpus.json`` relativo al directorio de trabajo y al directorio padre.

    Returns:
        Lista de documentos con las claves ``id``, ``titulo``, ``categoria`` y ``texto``.

    Raises:
        FileNotFoundError: Si no se localiza el archivo del corpus.
    """
    candidatas = [ruta] if ruta else [
        Path.cwd() / "corpus" / "corpus.json",
        Path.cwd().parent / "corpus" / "corpus.json",
    ]
    for candidata in candidatas:
        if candidata and candidata.exists():
            with candidata.open(encoding="utf-8") as archivo:
                return json.load(archivo)
    raise FileNotFoundError(
        "No se encontró corpus/corpus.json. Genéralo con: python corpus/generar_corpus.py"
    )


CORPUS = cargar_corpus()
print(f"Documentos cargados: {len(CORPUS)}")
print(f"Temáticas: {len(set(d['categoria'] for d in CORPUS))}")

Documentos cargados: 120
Temáticas: 12


In [ ]:
df_corpus = pd.DataFrame(CORPUS)
df_corpus["palabras"] = df_corpus["texto"].str.split().str.len()

resumen = (
    df_corpus.groupby("categoria")
    .agg(documentos=("id", "count"), palabras_promedio=("palabras", "mean"))
    .round(1)
    .sort_values("documentos", ascending=False)
)
print(resumen.to_string())
print(f"\nTotal de palabras en el corpus: {int(df_corpus['palabras'].sum()):,}")
print(f"Longitud media (palabras crudas): {df_corpus['palabras'].mean():.1f}")

                         documentos  palabras_promedio
categoria                                             
Arte y Música                    10               49.8
Astronomía                       10               59.7
Ciberseguridad                   10               49.8
Deportes                         10               53.4
Economía                         10               51.7
Gastronomía                      10               53.7
Historia                         10               51.1
Inteligencia Artificial          10               64.1
Medio Ambiente                   10               52.1
Salud                            10               53.8
Transporte                       10               50.1
Viajes y Geografía               10               47.6

Total de palabras en el corpus: 6,369
Longitud media (palabras crudas): 53.1


In [ ]:
# Vistazo a tres documentos de temáticas distintas.
for doc in (CORPUS[3], CORPUS[42], CORPUS[100]):
    print(f"[{doc['id']}] ({doc['categoria']}) {doc['titulo']}")
    print(f"    {doc['texto'][:180]}...\n")

[doc_003] (Inteligencia Artificial) Recuperación de información y RAG
    La generación aumentada por recuperación combina un buscador con un modelo generativo. Primero se recuperan los fragmentos más relevantes de una base documental y después se entreg...

[doc_042] (Deportes) Baloncesto y estadística avanzada
    El análisis de datos cambió la forma de jugar baloncesto. Al comparar la eficiencia por posesión, los equipos descubrieron que el triple y el tiro cercano al aro rinden más que el ...

[doc_100] (Ciberseguridad) Phishing e ingeniería social
    El phishing suplanta a una entidad de confianza para inducir a la víctima a entregar credenciales o a ejecutar un archivo malicioso. Las campañas dirigidas investigan a la persona ...



## 2. Preprocesamiento del texto

**Paso 2 de la actividad.** La función `preprocesar()` encadena cinco etapas, en este orden:

1. **Normalización a minúsculas** — `Búsqueda` y `búsqueda` deben ser el mismo término.
2. **Tokenización** — expresión regular que conserva letras (incluidas `á é í ó ú ü ñ`) y dígitos,
   y descarta implícitamente signos de puntuación, comillas y símbolos.
3. **Filtro de longitud** — se eliminan los tokens de un solo carácter, que no aportan señal.
4. **Eliminación de *stopwords*** — artículos, preposiciones y verbos auxiliares. Aparecen en casi
   todos los documentos, así que su poder discriminante es nulo y además inflan el índice.
5. ***Stemming*** — `Snowball` para español (algoritmo de Porter adaptado). Reduce
   *buscador*, *buscadores*, *búsquedas* y *buscar* a una raíz común, de modo que una consulta
   recupere documentos aunque la palabra aparezca flexionada de otra forma.

Como paso final se eliminan las tildes de la raíz, para que `eléctrico` y `electrico` colapsen en
el mismo término de índice. **La misma función se aplica al corpus y a la consulta**: es la
condición indispensable para que los términos sean comparables.

In [ ]:
# Lista de respaldo, usada si nltk no está instalado o no puede descargar su corpus.
STOPWORDS_RESPALDO = {
    "a", "al", "algo", "algunas", "algunos", "ante", "antes", "aquel", "aquella", "aquello",
    "aqui", "aquí", "asi", "así", "aunque", "bajo", "bien", "cada", "casi", "como", "cómo",
    "con", "contra", "cual", "cuales", "cuando", "cuanto", "de", "del", "desde", "donde",
    "dos", "durante", "e", "el", "él", "ella", "ellas", "ellos", "en", "entre", "era", "eran",
    "eres", "es", "esa", "esas", "ese", "eso", "esos", "esta", "está", "están", "estas", "este",
    "esto", "estos", "estar", "fue", "fueron", "ha", "haber", "hacia", "han", "hasta", "hay",
    "incluso", "la", "las", "le", "les", "lo", "los", "mas", "más", "me", "mediante", "mi",
    "mientras", "mucho", "muchos", "muy", "ni", "no", "nos", "o", "otra", "otras", "otro",
    "otros", "para", "pero", "poco", "por", "porque", "pues", "que", "qué", "quien", "se",
    "sea", "sean", "segun", "según", "ser", "si", "sí", "sido", "siendo", "sin", "sobre",
    "solo", "sólo", "son", "su", "sus", "tambien", "también", "tan", "tanto", "te", "tiene",
    "tienen", "toda", "todas", "todo", "todos", "tras", "tu", "tus", "un", "una", "uno",
    "unos", "y", "ya", "yo",
    # Formas verbales auxiliares (ser, estar, haber, tener), sin valor discriminante.
    "estaba", "estaban", "estamos", "estan", "están", "estas", "estás", "esté", "estén",
    "estuvo", "fuera", "fueran", "fuese", "fui", "fuimos", "fuiste", "ha", "habia", "había",
    "habian", "habían", "habrá", "hace", "hacen", "hacer", "hacía", "hago", "han", "has",
    "hemos", "hizo", "hubo", "puede", "pueden", "pudo", "sera", "será", "seran", "serán",
    "tenemos", "tenga", "tengo", "tenia", "tenía", "tuvo", "va", "van", "vez", "veces",
}


def cargar_stopwords() -> tuple[set[str], str]:
    """Obtiene el conjunto de palabras vacías en español.

    Intenta usar el corpus de ``nltk`` (descargándolo si hace falta) y cae en una lista
    interna cuando la librería o la descarga no están disponibles.

    Returns:
        Tupla ``(stopwords, origen)`` donde ``origen`` describe la fuente utilizada.
    """
    if HAY_NLTK:
        try:
            from nltk.corpus import stopwords

            try:
                palabras = stopwords.words("spanish")
            except LookupError:
                import io

                # El intérprete de python.org en macOS no usa el almacén de certificados del
                # sistema y la descarga falla por verificación TLS; certifi lo resuelve.
                try:
                    import certifi
                    import ssl

                    ssl._create_default_https_context = lambda *a, **k: ssl.create_default_context(
                        cafile=certifi.where()
                    )
                except ImportError:
                    pass
                # `print_error_to` evita que un fallo de descarga ensucie la salida: si no hay
                # red, el respaldo cubre el caso sin ruido.
                nltk.download("stopwords", quiet=True, print_error_to=io.StringIO())
                palabras = stopwords.words("spanish")
            return set(palabras) | STOPWORDS_RESPALDO, "nltk.corpus.stopwords('spanish')"
        except Exception:  # pragma: no cover - red o datos no disponibles
            pass
    return STOPWORDS_RESPALDO, "lista de respaldo integrada en el notebook"


STOPWORDS, ORIGEN_STOPWORDS = cargar_stopwords()
print(f"Stopwords: {len(STOPWORDS)} términos · origen: {ORIGEN_STOPWORDS}")

Stopwords: 368 términos · origen: nltk.corpus.stopwords('spanish')


In [ ]:
SUFIJOS_RESPALDO = (
    "amientos", "imientos", "amiento", "imiento", "aciones", "adoras", "adores", "ancias",
    "logias", "encias", "mente", "idades", "anzas", "ismos", "ables", "ibles", "istas",
    "ación", "acion", "adora", "ador", "ancia", "logia", "encia", "idad", "anza", "ismo",
    "able", "ible", "ista", "osos", "osas", "ivas", "ivos", "aron", "ando", "iendo", "ados",
    "idos", "amos", "emos", "imos", "ción", "cion", "ante", "oso", "osa", "iva", "ivo", "ada",
    "ado", "ida", "ido", "ar", "er", "ir", "as", "os", "es", "an", "en", "a", "e", "o", "s",
)


def stem_respaldo(palabra: str) -> str:
    """Reduce una palabra a su raíz recortando el sufijo español más largo aplicable.

    Solo se usa cuando ``nltk`` no está disponible. Es deliberadamente conservador:
    nunca deja una raíz de menos de cuatro caracteres.

    Args:
        palabra: Token ya normalizado a minúsculas.

    Returns:
        La raíz aproximada del token.
    """
    for sufijo in SUFIJOS_RESPALDO:
        if palabra.endswith(sufijo) and len(palabra) - len(sufijo) >= 4:
            return palabra[: -len(sufijo)]
    return palabra


def crear_stemmer() -> tuple[Callable[[str], str], str]:
    """Devuelve la función de *stemming* a utilizar y una etiqueta con su origen."""
    if HAY_NLTK:
        try:
            from nltk.stem import SnowballStemmer

            return SnowballStemmer("spanish").stem, "nltk SnowballStemmer('spanish')"
        except Exception:  # pragma: no cover
            pass
    return stem_respaldo, "stemmer de respaldo (recorte de sufijos)"


STEM, ORIGEN_STEMMER = crear_stemmer()
print(f"Stemmer: {ORIGEN_STEMMER}")
for ejemplo in ["buscadores", "búsquedas", "buscando", "eléctricos", "computadoras", "médicas"]:
    print(f"  {ejemplo:>14} -> {STEM(ejemplo)}")

Stemmer: nltk SnowballStemmer('spanish')
      buscadores -> buscador
       búsquedas -> busqued
        buscando -> busc
      eléctricos -> electr
    computadoras -> comput
         médicas -> medic


In [ ]:
PATRON_TOKEN = re.compile(r"[a-záéíóúüñ0-9]+")


def quitar_acentos(texto: str) -> str:
    """Elimina las tildes de un texto conservando la ``ñ``.

    Args:
        texto: Cadena de entrada en minúsculas.

    Returns:
        La cadena sin marcas diacríticas salvo la virgulilla de la ``ñ``.
    """
    protegido = texto.replace("ñ", "\0")
    descompuesto = unicodedata.normalize("NFD", protegido)
    sin_tildes = "".join(c for c in descompuesto if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", sin_tildes).replace("\0", "ñ")


def preprocesar(texto: str, devolver_etapas: bool = False) -> list[str] | dict[str, list[str]]:
    """Normaliza un texto libre a la lista de términos que alimenta el índice.

    Etapas: minúsculas -> tokenización (descarta puntuación) -> filtro de longitud ->
    eliminación de *stopwords* -> *stemming* -> eliminación de tildes de la raíz.

    Args:
        texto: Texto crudo del documento o de la consulta.
        devolver_etapas: Si es ``True`` devuelve el resultado intermedio de cada etapa,
            útil para ilustrar el proceso; si es ``False`` devuelve solo los términos finales.

    Returns:
        La lista de términos indexables, o un diccionario etapa -> tokens.
    """
    minusculas = texto.lower()
    tokens = PATRON_TOKEN.findall(minusculas)
    largos = [t for t in tokens if len(t) > 1]
    sin_vacias = [t for t in largos if t not in STOPWORDS]
    raices = [quitar_acentos(STEM(t)) for t in sin_vacias]
    if not devolver_etapas:
        return raices
    return {
        "1_minusculas": [minusculas],
        "2_tokenizacion": tokens,
        "3_filtro_longitud": largos,
        "4_sin_stopwords": sin_vacias,
        "5_stemming": raices,
    }

In [ ]:
# Demostración etapa por etapa del preprocesamiento.
EJEMPLO = (
    "Los AUTOMÓVILES eléctricos, ¿son realmente más limpios? "
    "Las baterías de iones de litio se cargan en 30 minutos."
)
etapas = preprocesar(EJEMPLO, devolver_etapas=True)
print(f"Texto original:\n  {EJEMPLO}\n")
for nombre, valores in etapas.items():
    etiqueta = nombre.split("_", 1)[1].replace("_", " ")
    print(f"{etiqueta:>16} ({len(valores):>2}): {valores}")

Texto original:
  Los AUTOMÓVILES eléctricos, ¿son realmente más limpios? Las baterías de iones de litio se cargan en 30 minutos.

      minusculas ( 1): ['los automóviles eléctricos, ¿son realmente más limpios? las baterías de iones de litio se cargan en 30 minutos.']
    tokenizacion (18): ['los', 'automóviles', 'eléctricos', 'son', 'realmente', 'más', 'limpios', 'las', 'baterías', 'de', 'iones', 'de', 'litio', 'se', 'cargan', 'en', '30', 'minutos']
 filtro longitud (18): ['los', 'automóviles', 'eléctricos', 'son', 'realmente', 'más', 'limpios', 'las', 'baterías', 'de', 'iones', 'de', 'litio', 'se', 'cargan', 'en', '30', 'minutos']
   sin stopwords (10): ['automóviles', 'eléctricos', 'realmente', 'limpios', 'baterías', 'iones', 'litio', 'cargan', '30', 'minutos']
        stemming (10): ['automovil', 'electr', 'realment', 'limpi', 'bat', 'ion', 'liti', 'carg', '30', 'minut']


In [ ]:
# El corpus completo se preprocesa una sola vez y se reutiliza.
TOKENS_POR_DOC = {doc["id"]: preprocesar(f"{doc['titulo']}. {doc['texto']}") for doc in CORPUS}

total_tokens = sum(len(t) for t in TOKENS_POR_DOC.values())
vocabulario = {t for tokens in TOKENS_POR_DOC.values() for t in tokens}
print(f"Tokens indexables totales : {total_tokens:,}")
print(f"Vocabulario (únicos)      : {len(vocabulario):,}")
print(f"Reducción vs. texto crudo : {1 - total_tokens / int(df_corpus['palabras'].sum()):.1%}")

Tokens indexables totales : 3,900
Vocabulario (únicos)      : 1,940
Reducción vs. texto crudo : 38.8%


## 3. Índice invertido

**Paso 3 de la actividad.** El índice invertido da la vuelta a la relación natural
*documento → palabras* para obtener *palabra → documentos*, que es la que necesita un buscador.

Estructura implementada con diccionarios nativos de Python:

```
postings = {
    "bm25":  {"doc_003": 2, "doc_017": 1},   # término -> {documento: tf}
    "bater": {"doc_100": 3, "doc_101": 4},
}
```

Junto a los *postings* el índice guarda las tres piezas que BM25 necesita después:

- `longitudes[doc]` — número de términos del documento tras el preprocesamiento (|D|).
- `avgdl` — longitud promedio de los documentos del corpus, usada para normalizar |D|.
- `df[t]` — número de documentos que contienen el término, base del IDF.

Consultar el índice cuesta O(1) por término más el recorrido de su lista de *postings*, en lugar
de recorrer los 120 documentos completos en cada búsqueda.

In [ ]:
class IndiceInvertido:
    """Índice invertido con frecuencias de término y estadísticas para el ranking.

    Attributes:
        documentos: Mapa ``id`` -> documento original.
        postings: Mapa ``término`` -> ``{id_documento: frecuencia}``.
        longitudes: Mapa ``id_documento`` -> número de términos del documento.
        avgdl: Longitud promedio de los documentos del corpus (``avgdl`` en BM25).
        n_docs: Número total de documentos indexados.
    """

    def __init__(self, documentos: Sequence[dict], tokens_por_doc: dict[str, list[str]]) -> None:
        """Construye el índice a partir de los documentos ya preprocesados.

        Args:
            documentos: Documentos originales, con al menos las claves ``id`` y ``texto``.
            tokens_por_doc: Términos preprocesados de cada documento, indexados por ``id``.
        """
        self.documentos: dict[str, dict] = {doc["id"]: doc for doc in documentos}
        self.postings: dict[str, dict[str, int]] = {}
        self.longitudes: dict[str, int] = {}

        for doc_id, tokens in tokens_por_doc.items():
            self.longitudes[doc_id] = len(tokens)
            for termino, frecuencia in Counter(tokens).items():
                self.postings.setdefault(termino, {})[doc_id] = frecuencia

        self.n_docs = len(self.documentos)
        self.avgdl = (
            sum(self.longitudes.values()) / self.n_docs if self.n_docs else 0.0
        )

    def df(self, termino: str) -> int:
        """Frecuencia documental: en cuántos documentos aparece el término."""
        return len(self.postings.get(termino, {}))

    def tf(self, termino: str, doc_id: str) -> int:
        """Frecuencia del término dentro de un documento concreto."""
        return self.postings.get(termino, {}).get(doc_id, 0)

    def documentos_con(self, termino: str) -> dict[str, int]:
        """Lista de *postings* del término: ``{id_documento: frecuencia}``."""
        return self.postings.get(termino, {})

    def vocabulario(self) -> list[str]:
        """Términos únicos del índice, ordenados alfabéticamente."""
        return sorted(self.postings)

    def __len__(self) -> int:
        """Tamaño del vocabulario indexado."""
        return len(self.postings)

    def __repr__(self) -> str:
        return (
            f"IndiceInvertido(documentos={self.n_docs}, vocabulario={len(self.postings)}, "
            f"avgdl={self.avgdl:.2f})"
        )


indice = IndiceInvertido(CORPUS, TOKENS_POR_DOC)
print(indice)

IndiceInvertido(documentos=120, vocabulario=1940, avgdl=32.50)


In [ ]:
# Estadísticas del índice construido.
frecuencias_globales = Counter(
    {t: sum(p.values()) for t, p in indice.postings.items()}
)
hapax = sum(1 for t in indice.postings if indice.df(t) == 1)

print(f"Documentos indexados        : {indice.n_docs}")
print(f"Términos únicos             : {len(indice):,}")
print(f"Entradas de postings         : {sum(len(p) for p in indice.postings.values()):,}")
print(f"Longitud promedio (avgdl)   : {indice.avgdl:.2f} términos")
print(f"Documento más corto / largo : {min(indice.longitudes.values())} / {max(indice.longitudes.values())}")
print(f"Términos en un solo documento (hapax): {hapax:,} ({hapax / len(indice):.1%})\n")

tabla_top = pd.DataFrame(
    [
        {"termino": t, "frecuencia_total": f, "df": indice.df(t)}
        for t, f in frecuencias_globales.most_common(12)
    ]
)
print("Términos más frecuentes del corpus:")
print(tabla_top.to_string(index=False))

Documentos indexados        : 120
Términos únicos             : 1,940
Entradas de postings         : 3,608
Longitud promedio (avgdl)   : 32.50 términos
Documento más corto / largo : 24 / 45
Términos en un solo documento (hapax): 1,211 (62.4%)

Términos más frecuentes del corpus:
termino  frecuencia_total  df
   part                18  17
    mil                17  15
  reduc                16  15
    agu                15  14
 permit                14  14
    dat                14   9
  model                14   9
recuper                13   8
  mayor                13  13
 sistem                12  12
   busc                12  12
 millon                12   9


In [ ]:
# Inspección de una lista de postings concreta.
def mostrar_postings(termino_crudo: str, limite: int = 6) -> None:
    """Imprime la lista de *postings* del término indicado, ya preprocesado.

    Args:
        termino_crudo: Palabra tal como la escribiría una persona.
        limite: Número máximo de documentos a mostrar.
    """
    raices = preprocesar(termino_crudo)
    if not raices:
        print(f"'{termino_crudo}' se eliminó en el preprocesamiento (stopword o token corto).")
        return
    raiz = raices[0]
    postings = indice.documentos_con(raiz)
    print(f"'{termino_crudo}' -> raíz '{raiz}' · df = {len(postings)} documentos")
    for doc_id, tf in sorted(postings.items(), key=lambda par: -par[1])[:limite]:
        doc = indice.documentos[doc_id]
        print(f"   {doc_id}  tf={tf:<2} |D|={indice.longitudes[doc_id]:<3} {doc['titulo']}")
    if len(postings) > limite:
        print(f"   ... y {len(postings) - limite} documentos más")


mostrar_postings("batería")
print()
mostrar_postings("estrellas")
print()
mostrar_postings("y")

'batería' -> raíz 'bat' · df = 3 documentos
   doc_091  tf=2  |D|=32  Baterías de estado sólido
   doc_061  tf=1  |D|=30  Energía solar fotovoltaica
   doc_090  tf=1  |D|=36  Automóviles eléctricos y autonomía

'estrellas' -> raíz 'estrell' · df = 3 documentos
   doc_014  tf=4  |D|=37  El ciclo de vida de las estrellas
   doc_011  tf=1  |D|=32  Agujeros negros supermasivos
   doc_013  tf=1  |D|=35  Exoplanetas y zonas habitables

'y' se eliminó en el preprocesamiento (stopword o token corto).


## 4. Algoritmo BM25

**Paso 4 de la actividad.** BM25 (*Best Matching 25*) puntúa cada documento $D$ frente a una
consulta $Q$ como la suma de la aportación de cada término $q_i$:

$$\text{score}(D, Q) \;=\; \sum_{i=1}^{n} \text{IDF}(q_i) \cdot
\frac{f(q_i, D)\,\bigl(k_1 + 1\bigr)}
{f(q_i, D) + k_1 \cdot \Bigl(1 - b + b \cdot \dfrac{|D|}{\text{avgdl}}\Bigr)}$$

$$\text{IDF}(q_i) \;=\; \ln\!\left(\frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1\right)$$

donde $f(q_i, D)$ es la frecuencia del término en el documento (`tf` del índice), $|D|$ su
longitud, **`avgdl` la longitud promedio de los documentos del corpus**, $N$ el número de
documentos y $n(q_i)$ la frecuencia documental (`df`).

Las tres ideas que hacen a BM25 mejor que un TF-IDF simple:

| Componente | Qué hace | Efecto |
|---|---|---|
| **IDF** | Penaliza términos presentes en muchos documentos | *batería* pesa más que *sistema* |
| **Saturación ($k_1$)** | La aportación de `tf` crece de forma asintótica hacia $k_1 + 1$ | El décimo *bm25* de un documento aporta mucho menos que el segundo |
| **Normalización por longitud ($b$)** | Divide $\lvert D\rvert$ entre `avgdl` | Un documento largo no gana por el mero hecho de contener más palabras |

Parámetros por defecto: $k_1 = 1.5$ (saturación moderada) y $b = 0.75$ (normalización parcial),
los valores estándar en Lucene y Elasticsearch. La interfaz del paso 5 permite moverlos y
observar el efecto en el ranking. El `+1` dentro del logaritmo es la variante que usa Lucene y
garantiza IDF no negativo incluso para términos que aparecen en más de la mitad del corpus.

In [ ]:
class BM25:
    """Ranking BM25 sobre un índice invertido.

    Attributes:
        indice: Índice invertido con los *postings* y las estadísticas del corpus.
        k1: Parámetro de saturación de la frecuencia de término (típico 1.2 – 2.0).
        b: Grado de normalización por longitud del documento (0 = ninguna, 1 = total).
    """

    def __init__(self, indice: IndiceInvertido, k1: float = 1.5, b: float = 0.75) -> None:
        """Inicializa el modelo y precalcula el IDF de todo el vocabulario.

        Args:
            indice: Índice invertido ya construido.
            k1: Parámetro de saturación.
            b: Parámetro de normalización por longitud.
        """
        self.indice = indice
        self.k1 = k1
        self.b = b
        self._idf = {termino: self._calcular_idf(termino) for termino in indice.postings}

    def _calcular_idf(self, termino: str) -> float:
        """IDF de Robertson en su variante de Lucene, siempre no negativa."""
        n_docs = self.indice.n_docs
        df = self.indice.df(termino)
        return math.log(1 + (n_docs - df + 0.5) / (df + 0.5))

    def idf(self, termino: str) -> float:
        """IDF del término; ``0.0`` si no está en el vocabulario del corpus."""
        return self._idf.get(termino, 0.0)

    def aporte(self, termino: str, doc_id: str) -> float:
        """Contribución de un único término a la puntuación de un documento.

        Args:
            termino: Término ya preprocesado.
            doc_id: Identificador del documento.

        Returns:
            El sumando de BM25 correspondiente a ese par término-documento.
        """
        tf = self.indice.tf(termino, doc_id)
        if tf == 0:
            return 0.0
        longitud = self.indice.longitudes[doc_id]
        normalizacion = 1 - self.b + self.b * longitud / self.indice.avgdl
        return self.idf(termino) * (tf * (self.k1 + 1)) / (tf + self.k1 * normalizacion)

    def puntuar(self, terminos: Iterable[str], doc_id: str) -> float:
        """Puntuación BM25 completa de un documento frente a una lista de términos."""
        return sum(self.aporte(termino, doc_id) for termino in terminos)

    def buscar(self, consulta: str, top_k: int = 5) -> list[dict]:
        """Recupera y ordena los documentos más relevantes para una consulta.

        Solo se puntúan los documentos que contienen al menos un término de la consulta
        (unión de las listas de *postings*), no los 120 del corpus.

        Args:
            consulta: Texto libre escrito por la persona usuaria.
            top_k: Número de resultados a devolver.

        Returns:
            Lista de resultados ordenada por puntuación descendente. Cada elemento incluye
            ``doc``, ``score``, ``terminos_encontrados`` y el desglose ``aportes``.
        """
        terminos = preprocesar(consulta)
        if not terminos:
            return []

        candidatos: set[str] = set()
        for termino in terminos:
            candidatos.update(self.indice.documentos_con(termino))

        resultados = []
        for doc_id in candidatos:
            aportes = {t: self.aporte(t, doc_id) for t in terminos}
            resultados.append(
                {
                    "doc": self.indice.documentos[doc_id],
                    "score": sum(aportes.values()),
                    "terminos_consulta": terminos,
                    "terminos_encontrados": [t for t, v in aportes.items() if v > 0],
                    "aportes": aportes,
                }
            )

        resultados.sort(key=lambda r: (-r["score"], r["doc"]["id"]))
        return resultados[:top_k]

    def explicar(self, consulta: str, doc_id: str) -> pd.DataFrame:
        """Desglosa la puntuación de un documento término a término.

        Args:
            consulta: Consulta en texto libre.
            doc_id: Documento a explicar.

        Returns:
            ``DataFrame`` con ``tf``, ``df``, ``idf`` y el aporte de cada término.
        """
        filas = []
        for termino in preprocesar(consulta):
            filas.append(
                {
                    "termino": termino,
                    "tf": self.indice.tf(termino, doc_id),
                    "df": self.indice.df(termino),
                    "idf": round(self.idf(termino), 4),
                    "aporte_bm25": round(self.aporte(termino, doc_id), 4),
                }
            )
        tabla = pd.DataFrame(filas)
        total = pd.DataFrame(
            [{"termino": "TOTAL", "tf": "", "df": "", "idf": "", "aporte_bm25": round(tabla["aporte_bm25"].sum(), 4)}]
        )
        return pd.concat([tabla, total], ignore_index=True)

    def __repr__(self) -> str:
        return f"BM25(k1={self.k1}, b={self.b}, N={self.indice.n_docs}, avgdl={self.indice.avgdl:.2f})"


bm25 = BM25(indice, k1=1.5, b=0.75)
print(bm25)

BM25(k1=1.5, b=0.75, N=120, avgdl=32.50)


In [ ]:
# Verificación numérica: la implementación contra el cálculo manual de la fórmula.
consulta_prueba = "energía solar fotovoltaica"
mejor = bm25.buscar(consulta_prueba, top_k=1)[0]
doc_id = mejor["doc"]["id"]

manual = 0.0
for termino in preprocesar(consulta_prueba):
    tf = indice.tf(termino, doc_id)
    if tf == 0:
        continue
    df = indice.df(termino)
    idf = math.log(1 + (indice.n_docs - df + 0.5) / (df + 0.5))
    norma = 1 - 0.75 + 0.75 * indice.longitudes[doc_id] / indice.avgdl
    manual += idf * (tf * (1.5 + 1)) / (tf + 1.5 * norma)

print(f"Consulta: '{consulta_prueba}' · documento {doc_id} ({mejor['doc']['titulo']})")
print(f"  score de la clase BM25 : {mejor['score']:.6f}")
print(f"  score calculado a mano : {manual:.6f}")
print(f"  ¿coinciden? {math.isclose(mejor['score'], manual, rel_tol=1e-12)}")

Consulta: 'energía solar fotovoltaica' · documento doc_061 (Energía solar fotovoltaica)
  score de la clase BM25 : 14.083525
  score calculado a mano : 14.083525
  ¿coinciden? True


In [ ]:
# Desglose término a término de esa misma puntuación.
print(f"Desglose de '{consulta_prueba}' sobre {doc_id}:\n")
print(bm25.explicar(consulta_prueba, doc_id).to_string(index=False))

Desglose de 'energía solar fotovoltaica' sobre doc_061:

 termino tf df     idf  aporte_bm25
   energ  2  6   2.924       4.2830
     sol  3  5   3.091       5.2528
fotovolt  1  1  4.3903       4.5477
   TOTAL                    14.0835


In [ ]:
# Efecto de la normalización por longitud: mismo término, documentos de distinta extensión.
raiz = preprocesar("energía")[0]
filas = []
for doc_id_i, tf in sorted(indice.documentos_con(raiz).items(), key=lambda p: -p[1])[:6]:
    filas.append(
        {
            "documento": doc_id_i,
            "titulo": indice.documentos[doc_id_i]["titulo"][:40],
            "tf": tf,
            "|D|": indice.longitudes[doc_id_i],
            "|D|/avgdl": round(indice.longitudes[doc_id_i] / indice.avgdl, 2),
            "aporte_b=0.75": round(bm25.aporte(raiz, doc_id_i), 4),
            "aporte_b=0": round(BM25(indice, k1=1.5, b=0.0).aporte(raiz, doc_id_i), 4),
        }
    )
print(f"Término '{raiz}' · avgdl = {indice.avgdl:.2f}\n")
print(pd.DataFrame(filas).to_string(index=False))

Término 'energ' · avgdl = 32.50

documento                             titulo  tf  |D|  |D|/avgdl  aporte_b=0.75  aporte_b=0
  doc_015    Materia oscura y energía oscura   2   37       1.14         3.9991      4.1771
  doc_061         Energía solar fotovoltaica   2   30       0.92         4.2830      4.1771
  doc_062              Energía eólica marina   1   33       1.02         2.9039      2.9240
  doc_063  Escasez de agua y gestión hídrica   1   33       1.02         2.9039      2.9240
  doc_090 Automóviles eléctricos y autonomía   1   36       1.11         2.7888      2.9240
  doc_099      Motores de combustión interna   1   33       1.02         2.9039      2.9240


## 5. Interfaz de consulta con `ipywidgets`

**Paso 5 de la actividad.** Interfaz dentro del notebook con caja de texto, botón de búsqueda y
controles para el número de resultados y los parámetros $k_1$ y $b$. La salida muestra el
**Top-5 de documentos con su puntuación BM25**, la categoría, el desglose de aportes por término
y un fragmento del documento con los términos coincidentes resaltados.

> Si `ipywidgets` no está instalado (`pip install ipywidgets`), la celda posterior ofrece la misma
> búsqueda en modo texto, de forma que el notebook siga siendo ejecutable de principio a fin.

In [ ]:
import html as _html


def resaltar(texto: str, terminos: set[str], maximo: int = 320) -> str:
    """Devuelve un fragmento del texto con los términos de la consulta resaltados.

    El resaltado se hace comparando la **raíz** de cada palabra del documento con las raíces de la
    consulta, de modo que se marcan también las variantes flexionadas.

    Args:
        texto: Texto original del documento.
        terminos: Raíces de la consulta que sí aparecen en el documento.
        maximo: Longitud aproximada del fragmento a mostrar.

    Returns:
        Fragmento en HTML, con las coincidencias envueltas en ``<mark>``.
    """
    palabras = re.findall(r"\w+|\W+", texto, flags=re.UNICODE)
    primera = next(
        (
            i
            for i, p in enumerate(palabras)
            if p.isalnum() and quitar_acentos(STEM(p.lower())) in terminos
        ),
        0,
    )
    inicio = max(0, primera - 25)
    trozo, largo = [], 0
    for palabra in palabras[inicio:]:
        if palabra.isalnum() and quitar_acentos(STEM(palabra.lower())) in terminos:
            trozo.append(f"<mark>{_html.escape(palabra)}</mark>")
        else:
            trozo.append(_html.escape(palabra))
        largo += len(palabra)
        if largo >= maximo:
            trozo.append(" …")
            break
    prefijo = "… " if inicio > 0 else ""
    return prefijo + "".join(trozo)


def resultados_a_html(consulta: str, resultados: list[dict], modelo: BM25) -> str:
    """Compone la tarjeta HTML con el ranking de resultados de una consulta."""
    if not resultados:
        return (
            f"<div style='font-family:system-ui;padding:12px;border-left:4px solid #c0392b;"
            f"background:#fdf0ef'><b>Sin resultados para «{_html.escape(consulta)}».</b><br>"
            f"Ningún documento contiene los términos de la consulta "
            f"({_html.escape(', '.join(preprocesar(consulta)) or 'consulta vacía tras el preprocesamiento')})."
            f"</div>"
        )

    partes = [
        "<div style='font-family:system-ui;line-height:1.45'>",
        f"<p style='margin:0 0 10px'>Consulta: <b>«{_html.escape(consulta)}»</b> · "
        f"términos indexados: <code>{_html.escape(' '.join(preprocesar(consulta)))}</code> · "
        f"k1={modelo.k1}, b={modelo.b}</p>",
    ]
    for posicion, resultado in enumerate(resultados, start=1):
        doc = resultado["doc"]
        encontrados = set(resultado["terminos_encontrados"])
        desglose = " + ".join(
            f"{t}:{v:.2f}" for t, v in sorted(resultado["aportes"].items(), key=lambda p: -p[1]) if v > 0
        )
        partes.append(
            "<div style='margin:0 0 12px;padding:10px 12px;border:1px solid #dcdcdc;"
            "border-radius:8px;background:#fafafa'>"
            f"<div style='display:flex;justify-content:space-between;gap:12px'>"
            f"<b>{posicion}. {_html.escape(doc['titulo'])}</b>"
            f"<span style='background:#1f6feb;color:#fff;padding:2px 10px;border-radius:12px;"
            f"font-variant-numeric:tabular-nums'>BM25 {resultado['score']:.4f}</span></div>"
            f"<div style='color:#555;font-size:0.88em;margin:4px 0 6px'>"
            f"{_html.escape(doc['id'])} · {_html.escape(doc['categoria'])} · "
            f"|D|={indice.longitudes[doc['id']]} términos · aportes: {_html.escape(desglose)}</div>"
            f"<div style='font-size:0.94em'>{resaltar(doc['texto'], encontrados)}</div>"
            "</div>"
        )
    partes.append("</div>")
    return "".join(partes)


def buscar_y_mostrar(consulta: str, top_k: int = 5, modelo: BM25 | None = None) -> None:
    """Ejecuta una búsqueda y muestra el resultado en HTML (o en texto si no hay IPython)."""
    modelo = modelo or bm25
    resultados = modelo.buscar(consulta, top_k=top_k)
    if HAY_WIDGETS:
        display(HTML(resultados_a_html(consulta, resultados, modelo)))
        return
    print(f"Consulta: «{consulta}» -> términos {preprocesar(consulta)}")
    if not resultados:
        print("  Sin resultados.")
        return
    for posicion, resultado in enumerate(resultados, start=1):
        doc = resultado["doc"]
        print(f"  {posicion}. [{resultado['score']:7.4f}] {doc['titulo']} ({doc['categoria']})")

In [ ]:
CONSULTAS_EJEMPLO = [
    "índice invertido y recuperación de información",
    "automóvil eléctrico batería de litio",
    "agujero negro supermasivo galaxia",
    "prevención de la diabetes y ejercicio",
    "masa madre fermentación pan",
    "inflación tasa de interés banco central",
    "ransomware respaldo y cifrado",
    "coche eléctrico",
]


def construir_interfaz(modelo: BM25) -> "widgets.Widget":
    """Construye la interfaz de búsqueda con ipywidgets.

    Args:
        modelo: Instancia de :class:`BM25` usada como referencia inicial; los deslizadores
            crean un modelo nuevo cuando se modifican ``k1`` o ``b``.

    Returns:
        El contenedor de widgets listo para mostrarse con ``display``.
    """
    caja_consulta = widgets.Text(
        value="automóvil eléctrico batería de litio",
        placeholder="Escribe tu consulta y presiona Enter…",
        description="Consulta:",
        continuous_update=False,
        layout=widgets.Layout(width="70%"),
        style={"description_width": "80px"},
    )
    ejemplos = widgets.Dropdown(
        options=[("— consultas de ejemplo —", "")] + [(c, c) for c in CONSULTAS_EJEMPLO],
        description="Ejemplos:",
        layout=widgets.Layout(width="70%"),
        style={"description_width": "80px"},
    )
    boton = widgets.Button(description="Buscar", button_style="primary", icon="search")
    top_k = widgets.IntSlider(value=5, min=1, max=10, description="Top-K:", continuous_update=False)
    k1 = widgets.FloatSlider(value=modelo.k1, min=0.0, max=3.0, step=0.1, description="k1:", continuous_update=False)
    b = widgets.FloatSlider(value=modelo.b, min=0.0, max=1.0, step=0.05, description="b:", continuous_update=False)
    salida = widgets.Output()

    def ejecutar(*_) -> None:
        """Lanza la búsqueda con los parámetros actuales y redibuja la salida."""
        with salida:
            salida.clear_output(wait=True)
            consulta = caja_consulta.value.strip()
            if not consulta:
                display(HTML("<i>Escribe una consulta para comenzar.</i>"))
                return
            activo = BM25(indice, k1=k1.value, b=b.value)
            resultados = activo.buscar(consulta, top_k=top_k.value)
            display(HTML(resultados_a_html(consulta, resultados, activo)))

    def usar_ejemplo(cambio: dict) -> None:
        """Copia la consulta de ejemplo seleccionada a la caja de texto y busca."""
        if cambio["new"]:
            caja_consulta.value = cambio["new"]
            ejecutar()

    boton.on_click(ejecutar)
    caja_consulta.observe(lambda _: ejecutar(), names="value")
    ejemplos.observe(usar_ejemplo, names="value")
    for control in (top_k, k1, b):
        control.observe(lambda _: ejecutar(), names="value")

    encabezado = widgets.HTML(
        "<h3 style='margin:0 0 4px'>🔎 Buscador BM25</h3>"
        f"<div style='color:#555'>Corpus de {indice.n_docs} documentos · "
        f"{len(indice):,} términos únicos · avgdl = {indice.avgdl:.1f}</div>"
    )
    ejecutar()
    return widgets.VBox(
        [encabezado, widgets.HBox([caja_consulta, boton]), ejemplos,
         widgets.HBox([top_k, k1, b]), salida]
    )


if HAY_WIDGETS:
    display(construir_interfaz(bm25))
else:
    print("ipywidgets no está instalado; usa buscar_y_mostrar('tu consulta') en la celda siguiente.")

> **Nota para la revisión.** La salida de un *widget* solo se renderiza en un kernel activo. Para
> que los resultados queden visibles en el archivo `.ipynb` entregado, las celdas siguientes
> repiten las mismas búsquedas de forma estática con `buscar_y_mostrar()`.

In [ ]:
buscar_y_mostrar("automóvil eléctrico batería de litio", top_k=5)

In [ ]:
buscar_y_mostrar("agujero negro supermasivo galaxia", top_k=5)

## 6. Evaluación: casos en los que el buscador acierta

Para medir el desempeño se define un conjunto de 14 consultas con la **categoría esperada** como
criterio de relevancia (*proxy* de juicio de relevancia). Sobre ese conjunto se calculan dos
métricas clásicas:

- **P@5** — proporción de los cinco primeros resultados que pertenecen a la categoría esperada.
  Es una métrica estricta: si la consulta solo recupera 3 documentos, los dos huecos cuentan como
  fallo. Por eso se reporta también **P@k efectiva**, calculada sobre los documentos realmente
  recuperados.
- **RR (*reciprocal rank*)** — inverso de la posición del primer resultado relevante; su promedio
  es el **MRR**, que premia colocar un acierto en la primera posición.

La categoría es además un criterio duro: la consulta *"energía solar y eólica renovable"* recupera
documentos de medio ambiente y de transporte, y estos últimos cuentan como fallo aunque hablen de
baterías y motores eléctricos. Las cifras son, por tanto, una **cota inferior** de la calidad real.

In [ ]:
CONSULTAS_EVALUACION: list[tuple[str, str]] = [
    ("índice invertido y recuperación de información", "Inteligencia Artificial"),
    ("redes neuronales profundas y retropropagación", "Inteligencia Artificial"),
    ("agujero negro supermasivo galaxia", "Astronomía"),
    ("telescopio espacial infrarrojo", "Astronomía"),
    ("prevención de la diabetes y ejercicio", "Salud"),
    ("resistencia a los antibióticos", "Salud"),
    ("masa madre fermentación pan", "Gastronomía"),
    ("nixtamalización del maíz para tortillas", "Gastronomía"),
    ("inflación tasa de interés banco central", "Economía"),
    ("ransomware respaldo y cifrado", "Ciberseguridad"),
    ("autenticación multifactor contraseñas", "Ciberseguridad"),
    ("automóvil eléctrico batería de litio", "Transporte"),
    ("energía solar y eólica renovable", "Medio Ambiente"),
    ("preparación para correr un maratón", "Deportes"),
]


def evaluar(modelo: BM25, consultas: Sequence[tuple[str, str]], k: int = 5) -> pd.DataFrame:
    """Calcula P@k y *reciprocal rank* para un conjunto de consultas etiquetadas.

    Args:
        modelo: Buscador a evaluar.
        consultas: Pares ``(consulta, categoría esperada)``.
        k: Profundidad del ranking a considerar.

    Returns:
        ``DataFrame`` con una fila por consulta y las columnas de métricas.
    """
    filas = []
    for consulta, esperada in consultas:
        resultados = modelo.buscar(consulta, top_k=k)
        aciertos = [r["doc"]["categoria"] == esperada for r in resultados]
        primer_acierto = next((i + 1 for i, ok in enumerate(aciertos) if ok), None)
        filas.append(
            {
                "consulta": consulta,
                "categoria_esperada": esperada,
                "recuperados": len(resultados),
                "relevantes": sum(aciertos),
                f"P@{k}": round(sum(aciertos) / k, 2),
                "P@k_efectiva": round(sum(aciertos) / len(resultados), 2) if resultados else 0.0,
                "1er_relevante": primer_acierto or "-",
                "RR": round(1 / primer_acierto, 3) if primer_acierto else 0.0,
                "top1": resultados[0]["doc"]["titulo"][:42] if resultados else "—",
                "score_top1": round(resultados[0]["score"], 3) if resultados else 0.0,
            }
        )
    return pd.DataFrame(filas)


evaluacion = evaluar(bm25, CONSULTAS_EVALUACION)
print(evaluacion.to_string(index=False))
print(
    f"\nP@5 promedio (estricta)      : {evaluacion['P@5'].mean():.3f}"
    f"\nP@k efectiva promedio        : {evaluacion['P@k_efectiva'].mean():.3f}"
    f"\nMRR                          : {evaluacion['RR'].mean():.3f}"
    f"\nPrimer resultado correcto    : "
    f"{(evaluacion['1er_relevante'] == 1).sum()} de {len(evaluacion)} consultas"
    f"\nDocumentos recuperados por consulta (mediana): {evaluacion['recuperados'].median():.0f}"
)
print(
    "\nLectura: el MRR de 1.0 indica que en las 14 consultas el documento correcto quedó en la "
    "primera posición. La P@5 estricta baja porque varias consultas recuperan menos de cinco "
    "documentos —el corpus solo tiene 10 por temática y BM25 no inventa resultados—, no porque "
    "el ranking se equivoque."
)

                                      consulta      categoria_esperada  recuperados  relevantes  P@5  P@k_efectiva  1er_relevante  RR                                 top1  score_top1
índice invertido y recuperación de información Inteligencia Artificial            5           2  0.4          0.40              1 1.0    Recuperación de información y RAG       7.228
 redes neuronales profundas y retropropagación Inteligencia Artificial            5           2  0.4          0.40              1 1.0           Redes neuronales profundas      16.788
             agujero negro supermasivo galaxia              Astronomía            5           4  0.8          0.80              1 1.0         Agujeros negros supermasivos      19.395
                telescopio espacial infrarrojo              Astronomía            3           1  0.2          0.33              1 1.0    El telescopio espacial James Webb      14.228
         prevención de la diabetes y ejercicio                   Salud            3  

In [ ]:
# Caso de acierto analizado en detalle: por qué gana el documento que gana.
CASO_ACIERTO = "nixtamalización del maíz para tortillas"
buscar_y_mostrar(CASO_ACIERTO, top_k=5)
ganador = bm25.buscar(CASO_ACIERTO, top_k=1)[0]["doc"]["id"]
print(f"\nDesglose de la puntuación del documento {ganador}:")
print(bm25.explicar(CASO_ACIERTO, ganador).to_string(index=False))
aporte_tf2 = bm25.aporte("nixtamaliz", ganador)
aporte_tf1 = bm25.aporte("tortill", ganador)
print(
    "\nLectura: los tres términos aparecen en un único documento del corpus (df = 1), así que "
    "comparten el IDF máximo y ningún otro documento del corpus puede competir. La diferencia "
    "entre sus aportes la marca el tf, amortiguada por la saturación de k1: pasar de tf=1 a tf=2 "
    f"sube el aporte de {aporte_tf1:.2f} a {aporte_tf2:.2f}, no a {2 * aporte_tf1:.2f}."
)


Desglose de la puntuación del documento doc_031:
   termino tf df     idf  aporte_bm25
nixtamaliz  2  1  4.3903       6.1802
      maiz  2  1  4.3903       6.1802
   tortill  1  1  4.3903       4.3010
     TOTAL                    16.6614

Lectura: los tres términos aparecen en un único documento del corpus (df = 1), así que comparten el IDF máximo y ningún otro documento del corpus puede competir. La diferencia entre sus aportes la marca el tf, amortiguada por la saturación de k1: pasar de tf=1 a tf=2 sube el aporte de 4.30 a 6.18, no a 8.60.


In [ ]:
# Segundo caso de acierto: consulta multitérmino donde el orden lo decide la acumulación de aportes.
CASO_ACIERTO_2 = "ransomware respaldo y cifrado"
buscar_y_mostrar(CASO_ACIERTO_2, top_k=5)
print(
    "Los tres términos coinciden en el documento de ransomware, mientras que los siguientes "
    "resultados solo aciertan uno o dos: la suma de aportes separa con claridad al primero."
)

Los tres términos coinciden en el documento de ransomware, mientras que los siguientes resultados solo aciertan uno o dos: la suma de aportes separa con claridad al primero.


In [ ]:
# Efecto de la saturación k1 y de la normalización b sobre el mismo ranking.
comparacion = []
for k1_v, b_v in [(1.5, 0.75), (1.5, 0.0), (0.0, 0.75), (3.0, 1.0)]:
    modelo = BM25(indice, k1=k1_v, b=b_v)
    metricas = evaluar(modelo, CONSULTAS_EVALUACION)
    comparacion.append(
        {
            "k1": k1_v,
            "b": b_v,
            "P@5": round(metricas["P@5"].mean(), 3),
            "MRR": round(metricas["RR"].mean(), 3),
        }
    )
print("Sensibilidad a los parámetros (k1=0 ignora la frecuencia; b=0 ignora la longitud):\n")
print(pd.DataFrame(comparacion).to_string(index=False))
print(
    "\nLas métricas agregadas apenas se mueven: con 120 documentos de longitud muy parecida "
    f"(entre {min(indice.longitudes.values())} y {max(indice.longitudes.values())} términos) la "
    "normalización tiene poco margen de actuación. El efecto sí es visible en el ranking concreto:"
)

CONSULTA_SENSIBILIDAD = "energía"
for k1_v, b_v in [(1.5, 0.75), (1.5, 0.0), (0.0, 0.75)]:
    modelo = BM25(indice, k1=k1_v, b=b_v)
    orden = [
        f"{r['doc']['titulo'][:34]} ({r['score']:.3f}, |D|={indice.longitudes[r['doc']['id']]})"
        for r in modelo.buscar(CONSULTA_SENSIBILIDAD, top_k=3)
    ]
    print(f"\n  k1={k1_v}, b={b_v} -> '{CONSULTA_SENSIBILIDAD}'")
    for posicion, linea in enumerate(orden, start=1):
        print(f"     {posicion}. {linea}")
print(
    "\nCon b=0.75 el documento corto gana; con b=0 los documentos empatan en el término y solo "
    "decide la frecuencia; con k1=0 la frecuencia deja de contar y el ranking se reduce al IDF."
)

Sensibilidad a los parámetros (k1=0 ignora la frecuencia; b=0 ignora la longitud):

 k1    b   P@5  MRR
1.5 0.75 0.500  1.0
1.5 0.00 0.500  1.0
0.0 0.75 0.514  1.0
3.0 1.00 0.500  1.0

Las métricas agregadas apenas se mueven: con 120 documentos de longitud muy parecida (entre 24 y 45 términos) la normalización tiene poco margen de actuación. El efecto sí es visible en el ranking concreto:

  k1=1.5, b=0.75 -> 'energía'
     1. Energía solar fotovoltaica (4.283, |D|=30)
     2. Materia oscura y energía oscura (3.999, |D|=37)
     3. Energía eólica marina (2.904, |D|=33)

  k1=1.5, b=0.0 -> 'energía'
     1. Materia oscura y energía oscura (4.177, |D|=37)
     2. Energía solar fotovoltaica (4.177, |D|=30)
     3. Energía eólica marina (2.924, |D|=33)

  k1=0.0, b=0.75 -> 'energía'
     1. Materia oscura y energía oscura (2.924, |D|=37)
     2. Energía solar fotovoltaica (2.924, |D|=30)
     3. Energía eólica marina (2.924, |D|=33)

Con b=0.75 el documento corto gana; con b=0 los document

## 7. Caso de fallo: el límite de la búsqueda exacta (sinónimos)

BM25 es un modelo **léxico**: puntúa la coincidencia de cadenas, no de significados. Si un término
de la consulta no está en el vocabulario del índice, su `df` es 0, su IDF se define como 0 y su
aportación al ranking es **exactamente cero**. El buscador no falla por error de cálculo: falla
porque para él *coche* y *automóvil* son dos símbolos sin relación alguna.

El corpus se redactó a propósito usando **automóvil**, **médico** y **computadora**, y evitando
**coche**, **doctor** y **ordenador**. Las celdas siguientes contrastan cada par.

In [ ]:
PARES_SINONIMOS = [
    ("coche eléctrico", "automóvil eléctrico"),
    ("doctor", "médico"),
    ("ordenador", "computadora"),
]

filas = []
for consulta_fallo, consulta_ok in PARES_SINONIMOS:
    for etiqueta, consulta in (("consulta fallida", consulta_fallo), ("sinónimo del corpus", consulta_ok)):
        terminos = preprocesar(consulta)
        resultados = bm25.buscar(consulta, top_k=5)
        filas.append(
            {
                "tipo": etiqueta,
                "consulta": consulta,
                "términos": " ".join(terminos),
                "df de cada término": " ".join(f"{t}={indice.df(t)}" for t in terminos),
                "resultados": len(resultados),
                "score_top1": round(resultados[0]["score"], 3) if resultados else 0.0,
                "top1": resultados[0]["doc"]["titulo"][:42] if resultados else "— SIN RESULTADOS —",
            }
        )
print(pd.DataFrame(filas).to_string(index=False))
score_fallo = bm25.buscar("coche eléctrico", top_k=1)[0]["score"]
score_ok = bm25.buscar("automóvil eléctrico", top_k=1)[0]["score"]
print(
    "\nTres modos de fallo distintos en la misma tabla:\n"
    "  1. 'coche eléctrico' -> devuelve resultados, pero el término clave aporta cero (df = 0) y "
    f"el orden lo decide solo 'electr': el score del mismo documento cae de {score_ok:.2f} a "
    f"{score_fallo:.2f}.\n"
    "  2. 'doctor' -> cero resultados, pese a que el corpus habla de médicos y de telemedicina.\n"
    "  3. 'ordenador' -> el peor caso, un FALSO POSITIVO: el stemmer lo reduce a 'orden', que "
    "colisiona con 'ordena/orden', y el primer resultado es un documento sobre chocolate. "
    "El stemming, que en general ayuda, aquí fabrica una coincidencia que no existe."
)

               tipo            consulta         términos   df de cada término  resultados  score_top1                               top1
   consulta fallida     coche eléctrico      coch electr      coch=0 electr=5           5       5.017 Automóviles eléctricos y autonomía
sinónimo del corpus automóvil eléctrico automovil electr automovil=2 electr=5           5      10.373 Automóviles eléctricos y autonomía
   consulta fallida              doctor           doctor             doctor=0           0       0.000                 — SIN RESULTADOS —
sinónimo del corpus              médico            medic              medic=1           1       4.187   Prevención de la diabetes tipo 2
   consulta fallida           ordenador            orden              orden=2           2       4.077           Chocolate desde el cacao
sinónimo del corpus         computadora           comput             comput=4           4       4.460             Visión por computadora

Tres modos de fallo distintos en la mism

In [ ]:
# El caso más ilustrativo: 'coche eléctrico' recupera documentos, pero por el motivo equivocado.
buscar_y_mostrar("coche eléctrico", top_k=5)
print("Aportes de cada término de la consulta al documento mejor puntuado:")
mejor_coche = bm25.buscar("coche eléctrico", top_k=1)[0]["doc"]["id"]
print(bm25.explicar("coche eléctrico", mejor_coche).to_string(index=False))
print(
    "\n'coch' tiene df = 0 y aporte 0.0: el ranking lo decide únicamente 'electr'. "
    "El documento sobre automóviles eléctricos —el que la persona buscaba— compite en igualdad "
    "de condiciones con documentos de energía solar o de baterías de estado sólido, y puede "
    "quedar por debajo."
)

Aportes de cada término de la consulta al documento mejor puntuado:
termino tf df    idf  aporte_bm25
   coch  0  0    0.0       0.0000
 electr  3  5  3.091       5.0167
  TOTAL                    5.0167

'coch' tiene df = 0 y aporte 0.0: el ranking lo decide únicamente 'electr'. El documento sobre automóviles eléctricos —el que la persona buscaba— compite en igualdad de condiciones con documentos de energía solar o de baterías de estado sólido, y puede quedar por debajo.


In [ ]:
# Caso extremo: cero resultados pese a que el corpus sí trata el tema.
buscar_y_mostrar("doctor", top_k=5)
print("La misma intención, con la palabra que sí usa el corpus:")
buscar_y_mostrar("médico", top_k=5)
print(
    "Consulta de una sola palabra: 'doctor' no está en el vocabulario, no hay ningún término que "
    "aporte puntuación y el conjunto de candidatos queda vacío. El corpus habla de médicos en "
    "varios documentos, pero el buscador no tiene forma de saberlo.\n"
)

# Peor todavía: en una consulta de varias palabras el hueco lo rellena cualquier otro término.
buscar_y_mostrar("cita con el doctor", top_k=3)
print(
    "'doctor' aporta cero y 'cita' se reduce a la raíz 'cit', que el corpus usa en "
    "«las cuatro grandes citas del calendario». El buscador responde con un documento de tenis "
    "y lo presenta con la misma confianza que cualquier acierto: no hay señal alguna de que la "
    "consulta se entendió mal."
)

La misma intención, con la palabra que sí usa el corpus:


Consulta de una sola palabra: 'doctor' no está en el vocabulario, no hay ningún término que aporte puntuación y el conjunto de candidatos queda vacío. El corpus habla de médicos en varios documentos, pero el buscador no tiene forma de saberlo.



'doctor' aporta cero y 'cita' se reduce a la raíz 'cit', que el corpus usa en «las cuatro grandes citas del calendario». El buscador responde con un documento de tenis y lo presenta con la misma confianza que cualquier acierto: no hay señal alguna de que la consulta se entendió mal.


### 7.1 Mitigación 1 — expansión de la consulta con un diccionario de sinónimos

La corrección más barata dentro del propio paradigma léxico: antes de puntuar, se añaden a la
consulta las variantes conocidas de cada término. No requiere reconstruir el índice, pero exige
mantener el diccionario a mano y no cubre sinónimos imprevistos ni relaciones semánticas
(*"vehículo de cero emisiones"* seguiría fallando).

In [ ]:
SINONIMOS: dict[str, list[str]] = {
    "coche": ["automovil", "vehiculo"],
    "carro": ["automovil", "vehiculo"],
    "auto": ["automovil"],
    "doctor": ["medico"],
    "doctora": ["medico"],
    "ordenador": ["computadora"],
    "pc": ["computadora"],
    "celular": ["telefono"],
    "trabajo": ["empleo"],
    "bateria": ["acumulador"],
}
SINONIMOS_INDEXADOS = {
    quitar_acentos(STEM(clave)): [quitar_acentos(STEM(v)) for v in valores]
    for clave, valores in SINONIMOS.items()
}


def expandir_consulta(consulta: str) -> list[str]:
    """Expande una consulta con los sinónimos conocidos de cada término.

    Args:
        consulta: Texto libre de la persona usuaria.

    Returns:
        Lista de términos preprocesados, con las variantes añadidas al final.
    """
    terminos = preprocesar(consulta)
    expandidos = list(terminos)
    for termino in terminos:
        for variante in SINONIMOS_INDEXADOS.get(termino, []):
            if variante not in expandidos:
                expandidos.append(variante)
    return expandidos


class BM25Expandido(BM25):
    """Variante de :class:`BM25` que expande la consulta con sinónimos antes de puntuar."""

    def buscar(self, consulta: str, top_k: int = 5) -> list[dict]:
        """Busca usando la consulta expandida; el resto del ranking es idéntico al de BM25."""
        terminos = expandir_consulta(consulta)
        if not terminos:
            return []
        candidatos: set[str] = set()
        for termino in terminos:
            candidatos.update(self.indice.documentos_con(termino))
        resultados = []
        for doc_id in candidatos:
            aportes = {t: self.aporte(t, doc_id) for t in terminos}
            resultados.append(
                {
                    "doc": self.indice.documentos[doc_id],
                    "score": sum(aportes.values()),
                    "terminos_consulta": terminos,
                    "terminos_encontrados": [t for t, v in aportes.items() if v > 0],
                    "aportes": aportes,
                }
            )
        resultados.sort(key=lambda r: (-r["score"], r["doc"]["id"]))
        return resultados[:top_k]


bm25_expandido = BM25Expandido(indice, k1=1.5, b=0.75)

comparativa = []
for consulta in ["coche eléctrico", "cita con el doctor", "ordenador"]:
    base = bm25.buscar(consulta, top_k=1)
    expandida = bm25_expandido.buscar(consulta, top_k=1)
    comparativa.append(
        {
            "consulta": consulta,
            "BM25 · top1": base[0]["doc"]["titulo"][:40] if base else "— sin resultados —",
            "BM25 · score": round(base[0]["score"], 3) if base else 0.0,
            "expandido · top1": expandida[0]["doc"]["titulo"][:40] if expandida else "— sin resultados —",
            "expandido · score": round(expandida[0]["score"], 3) if expandida else 0.0,
        }
    )
print(pd.DataFrame(comparativa).to_string(index=False))

          consulta                        BM25 · top1  BM25 · score                   expandido · top1  expandido · score
   coche eléctrico Automóviles eléctricos y autonomía         5.017 Automóviles eléctricos y autonomía             10.373
cita con el doctor       Tenis y superficies de juego         3.801   Prevención de la diabetes tipo 2              4.187
         ordenador           Chocolate desde el cacao         4.077             Visión por computadora              4.460


In [ ]:
print("Resultado de 'coche eléctrico' CON expansión de sinónimos:")
buscar_y_mostrar("coche eléctrico", top_k=5, modelo=bm25_expandido)

Resultado de 'coche eléctrico' CON expansión de sinónimos:


### 7.2 Mitigación 2 — recuperación densa e híbrida

La expansión manual no escala: hay que anticipar cada sinónimo. La alternativa estándar hoy es
representar consulta y documentos como **vectores densos** (*embeddings*), donde *coche* y
*automóvil* quedan próximos en el espacio semántico aunque no compartan ni una letra, y combinar
ambas señales:

| Enfoque | Resuelve sinónimos | Términos raros y exactos | Costo | Explicabilidad |
|---|---|---|---|---|
| **BM25 (léxico)** | No | Excelente (nombres propios, códigos, siglas) | Muy bajo, CPU | Total: se audita término a término |
| **Denso (*embeddings*)** | Sí | Débil con vocabulario fuera de distribución | Alto: modelo + índice vectorial | Baja |
| **Híbrido (BM25 + denso, fusión RRF)** | Sí | Buena | Medio-alto | Media |

Por eso BM25 sigue vigente treinta años después: no se sustituye, se combina. En una arquitectura
RAG el recuperador híbrido es hoy el estándar, y BM25 aporta justo lo que al vector denso le
cuesta — la coincidencia literal de un término técnico infrecuente.

## 8. Conclusiones

1. **El preprocesamiento decide el techo del sistema.** La cadena de normalización redujo el
   corpus de 6 369 palabras crudas a unos 3 900 términos indexables (≈ 38 %) y un vocabulario de
   cerca de 1 950 raíces. Aplicar exactamente la misma cadena a la consulta es lo que permite que
   *baterías* recupere documentos que escriben *batería*; cualquier asimetría entre ambos caminos
   rompe la recuperación de forma silenciosa y sin error visible.

2. **BM25 acierta de forma consistente cuando consulta y documento comparten vocabulario.** En las
   14 consultas evaluadas el **MRR fue 1.0**: el documento correcto quedó siempre en la primera
   posición. Los aciertos más nítidos son los de términos con `df` bajo (*nixtamalización*,
   *ransomware*, *fotovoltaica*): el IDF los premia y dominan la suma de aportes.

3. **Los tres componentes de la fórmula cumplen funciones distintas y comprobables.** En este
   corpus las métricas agregadas apenas se mueven al variar `k1` y `b`, porque los documentos
   tienen longitudes muy parecidas (24 a 45 términos); el efecto sí es visible en rankings
   concretos, donde `b` decide entre dos documentos con el mismo `tf` y distinta extensión.
   Conviene no sobreinterpretar una tabla de sensibilidad calculada sobre un corpus pequeño.

4. **El fallo no es un defecto de implementación, es el límite del modelo.** BM25 opera sobre
   cadenas, no sobre significados, y eso produjo tres modos de fallo distintos: aporte cero con
   resultados degradados (*coche eléctrico*), cero resultados (*doctor*) y —el más peligroso— un
   **falso positivo por colisión de raíces**, donde *ordenador* se reduce a *orden* y el primer
   resultado es un documento sobre chocolate. Los tres se diagnostican mirando el `df` de los
   términos de la consulta.

5. **La mitigación depende del presupuesto.** Un diccionario de sinónimos devolvió *coche
   eléctrico* a su puntuación correcta (de 5.02 a 10.37) con costo casi nulo y sin tocar el
   índice, pero solo cubre lo que alguien anticipó. La recuperación densa resuelve el caso
   general a cambio de un modelo, un índice vectorial y una pérdida de explicabilidad. La
   práctica actual combina ambas señales en un recuperador híbrido.

---

### Cómo reproducir este notebook

```bash
pip install -r requirements.txt
python corpus/generar_corpus.py          # regenera corpus/corpus.json (120 documentos)
jupyter lab motor_busqueda_bm25.ipynb    # Kernel -> Restart & Run All
```